# Analisis del dataset Bank Marketing

Este notebook realiza limpieza, analisis exploratorio, graficas, entrenamiento de modelos de machine learning y una conclusion ejecutiva para direccion.

**Objetivo:** predecir si un cliente suscribira un deposito a plazo (`y = yes`).

**Nota importante de negocio:** la variable `duration` solo se conoce despues de realizar la llamada. Por eso se entrenan dos escenarios:

- **Modelo operativo sin `duration`:** util para priorizar clientes antes de llamar.
- **Modelo post-llamada con `duration`:** util como diagnostico, pero no para scoring preventa.

## 1. Librerias

Si alguna libreria no esta instalada, ejecuta en una terminal:

```bash
pip install pandas numpy matplotlib seaborn scikit-learn openpyxl
```

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")

DATA_PATH = Path("bank-full.csv")
OUTPUT_DIR = Path("outputs_notebook")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 20260709

## 2. Carga y primera inspeccion

In [ ]:
df = pd.read_csv(DATA_PATH, sep=";")
df.head()

In [ ]:
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")
display(df.info())
display(df.describe(include="all").T)

## 3. Limpieza y calidad de datos

El dataset no suele traer nulos reales, pero si categorias `unknown`. En este analisis se mantienen como categoria informativa, porque en marketing la ausencia de informacion tambien puede tener valor predictivo.

In [ ]:
df_clean = df.copy()

numeric_cols = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]
categorical_cols = [
    "job", "marital", "education", "default", "housing", "loan",
    "contact", "month", "poutcome"
]
target_col = "y"

df_clean[target_col] = df_clean[target_col].map({"no": 0, "yes": 1})

quality = pd.DataFrame({
    "variable": df.columns,
    "nulos": df.isna().sum().values,
    "unknown": [(df[col] == "unknown").sum() if df[col].dtype == "object" else 0 for col in df.columns],
    "valores_unicos": df.nunique().values,
})
display(quality)

quality.to_csv(OUTPUT_DIR / "calidad_datos.csv", index=False)

## 4. Analisis exploratorio con graficas

In [ ]:
target_rate = df_clean[target_col].mean()
print(f"Tasa de suscripcion: {target_rate:.2%}")

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x="y", ax=ax)
ax.set_title("Distribucion de la variable objetivo")
ax.set_xlabel("Suscripcion")
ax.set_ylabel("Clientes")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_distribucion_objetivo.png", dpi=160)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, col in zip(axes.ravel(), ["age", "balance", "duration", "campaign"]):
    sns.histplot(data=df, x=col, hue="y", bins=40, kde=False, ax=ax)
    ax.set_title(f"Distribucion de {col} por respuesta")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "02_distribuciones_numericas.png", dpi=160)
plt.show()

In [ ]:
def plot_rate_by_category(data, column, top=None):
    rates = (
        data.groupby(column)[target_col]
        .agg(tasa="mean", clientes="count")
        .sort_values("tasa", ascending=False)
        .reset_index()
    )
    if top:
        rates = rates.head(top)

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=rates, x="tasa", y=column, ax=ax)
    ax.set_title(f"Tasa de suscripcion por {column}")
    ax.set_xlabel("Tasa de suscripcion")
    ax.set_ylabel(column)
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"03_tasa_por_{column}.png", dpi=160)
    plt.show()
    return rates

rates_month = plot_rate_by_category(df_clean, "month")
rates_poutcome = plot_rate_by_category(df_clean, "poutcome")
rates_job = plot_rate_by_category(df_clean, "job")

In [ ]:
corr = df_clean[numeric_cols + [target_col]].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(corr, annot=True, cmap="vlag", center=0, fmt=".2f", ax=ax)
ax.set_title("Correlaciones entre variables numericas")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_correlaciones.png", dpi=160)
plt.show()

## 5. Preparacion para modelado

Se usa `OneHotEncoder` para variables categoricas y `StandardScaler` para variables numericas en regresion logistica. Para random forest y gradient boosting, se mantiene el mismo preprocesamiento por simplicidad y comparabilidad.

In [ ]:
def make_preprocessor(feature_cols):
    num_features = [c for c in numeric_cols if c in feature_cols]
    cat_features = [c for c in categorical_cols if c in feature_cols]

    numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, num_features),
            ("cat", categorical_pipe, cat_features),
        ]
    )


def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)[:, 1]
    else:
        proba = model.decision_function(X_test)

    pred = (proba >= 0.5).astype(int)

    return {
        "modelo": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "auc": roc_auc_score(y_test, proba),
        "matriz_confusion": confusion_matrix(y_test, pred),
        "proba": proba,
        "pipeline": model,
    }


def train_scenario(df_model, use_duration=False):
    scenario_name = "con_duration" if use_duration else "sin_duration"
    features = numeric_cols + categorical_cols
    if not use_duration:
        features = [c for c in features if c != "duration"]

    X = df_model[features]
    y = df_model[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    preprocessor = make_preprocessor(features)

    models = {
        f"Regresion logistica {scenario_name}": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        f"Random forest {scenario_name}": RandomForestClassifier(
            n_estimators=250,
            max_depth=12,
            min_samples_leaf=25,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        f"Gradient boosting {scenario_name}": GradientBoostingClassifier(
            random_state=RANDOM_STATE,
        ),
    }

    results = []
    for name, estimator in models.items():
        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ])
        results.append(evaluate_model(name, pipe, X_train, X_test, y_train, y_test))

    return results, X_test, y_test


results_no_duration, X_test_no_duration, y_test_no_duration = train_scenario(df_clean, use_duration=False)
results_with_duration, X_test_with_duration, y_test_with_duration = train_scenario(df_clean, use_duration=True)

all_results = results_no_duration + results_with_duration

## 6. Comparacion de modelos

In [ ]:
metrics_df = pd.DataFrame([
    {
        "modelo": r["modelo"],
        "accuracy": r["accuracy"],
        "precision": r["precision"],
        "recall": r["recall"],
        "f1": r["f1"],
        "auc": r["auc"],
    }
    for r in all_results
]).sort_values("auc", ascending=False)

display(metrics_df.style.format({
    "accuracy": "{:.2%}",
    "precision": "{:.2%}",
    "recall": "{:.2%}",
    "f1": "{:.3f}",
    "auc": "{:.3f}",
})))

metrics_df.to_csv(OUTPUT_DIR / "comparacion_modelos.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
plot_df = metrics_df.melt(id_vars="modelo", value_vars=["auc", "f1", "precision", "recall"], var_name="metrica", value_name="valor")
sns.barplot(data=plot_df, x="valor", y="modelo", hue="metrica", ax=ax)
ax.set_title("Comparacion de modelos")
ax.set_xlabel("Valor")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "05_comparacion_modelos.png", dpi=160)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for result in all_results:
    y_test = y_test_with_duration if "con_duration" in result["modelo"] else y_test_no_duration
    RocCurveDisplay.from_predictions(y_test, result["proba"], name=result["modelo"], ax=ax)
ax.set_title("Curvas ROC")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "06_curvas_roc.png", dpi=160)
plt.show()

## 7. Mejor modelo operativo y matriz de confusion

Para decision comercial previa a la llamada, solo se comparan modelos **sin `duration`**.

In [ ]:
best_operational = max(results_no_duration, key=lambda r: r["auc"])
print("Mejor modelo operativo:", best_operational["modelo"])
print(f"AUC: {best_operational['auc']:.3f}")
print(f"Precision: {best_operational['precision']:.2%}")
print(f"Recall: {best_operational['recall']:.2%}")
print(f"F1: {best_operational['f1']:.3f}")

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(best_operational["matriz_confusion"], annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title(f"Matriz de confusion - {best_operational['modelo']}")
ax.set_xlabel("Prediccion")
ax.set_ylabel("Real")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "07_matriz_confusion_operativa.png", dpi=160)
plt.show()

print(classification_report(y_test_no_duration, (best_operational["proba"] >= 0.5).astype(int), target_names=["no", "yes"]))

## 8. Lift por decil

El lift permite evaluar si el modelo concentra conversiones en los clientes de mayor score. Esto es muy util para priorizar campanas cuando el presupuesto o la capacidad comercial son limitados.

In [ ]:
def lift_table(y_true, proba, n_bins=10):
    tmp = pd.DataFrame({"y": y_true.values, "score": proba})
    tmp = tmp.sort_values("score", ascending=False).reset_index(drop=True)
    tmp["decil"] = pd.qcut(tmp.index + 1, q=n_bins, labels=[f"D{i}" for i in range(1, n_bins + 1)])
    base_rate = tmp["y"].mean()
    lift = tmp.groupby("decil").agg(
        clientes=("y", "count"),
        suscripciones=("y", "sum"),
        tasa=("y", "mean"),
        score_min=("score", "min"),
        score_max=("score", "max"),
    ).reset_index()
    lift["lift"] = lift["tasa"] / base_rate
    return lift

lift_df = lift_table(y_test_no_duration, best_operational["proba"])
display(lift_df.style.format({"tasa": "{:.2%}", "lift": "{:.2f}", "score_min": "{:.3f}", "score_max": "{:.3f}"}))
lift_df.to_csv(OUTPUT_DIR / "lift_modelo_operativo.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=lift_df, x="decil", y="lift", ax=ax)
ax.axhline(1, color="red", linestyle="--", label="Promedio")
ax.set_title("Lift por decil - modelo operativo")
ax.set_xlabel("Decil de score, de mayor a menor")
ax.set_ylabel("Lift")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "08_lift_operativo.png", dpi=160)
plt.show()

## 9. Importancia de variables

Para modelos de arboles se puede consultar la importancia de variables ya codificadas. Las variables categoricas aparecen como columnas dummy despues del one-hot encoding.

In [ ]:
def get_feature_names(preprocessor):
    num_names = preprocessor.named_transformers_["num"].feature_names_in_.tolist()
    cat_pipe = preprocessor.named_transformers_["cat"]
    cat_names = cat_pipe.named_steps["onehot"].get_feature_names_out(cat_pipe.feature_names_in_).tolist()
    return num_names + cat_names

best_pipe = best_operational["pipeline"]
model = best_pipe.named_steps["model"]
preprocessor = best_pipe.named_steps["preprocessor"]

if hasattr(model, "feature_importances_"):
    feature_names = get_feature_names(preprocessor)
    importances = pd.DataFrame({
        "variable": feature_names,
        "importancia": model.feature_importances_,
    }).sort_values("importancia", ascending=False).head(20)

    display(importances)
    importances.to_csv(OUTPUT_DIR / "importancia_variables.csv", index=False)

    fig, ax = plt.subplots(figsize=(10, 7))
    sns.barplot(data=importances, x="importancia", y="variable", ax=ax)
    ax.set_title("Top 20 variables mas importantes")
    ax.set_xlabel("Importancia")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "09_importancia_variables.png", dpi=160)
    plt.show()
else:
    print("El mejor modelo operativo no expone feature_importances_.")

## 10. Exportar resultados

In [ ]:
with pd.ExcelWriter(OUTPUT_DIR / "resultados_bank_marketing_notebook.xlsx") as writer:
    quality.to_excel(writer, sheet_name="Calidad datos", index=False)
    metrics_df.to_excel(writer, sheet_name="Comparacion modelos", index=False)
    lift_df.to_excel(writer, sheet_name="Lift operativo", index=False)
    rates_month.to_excel(writer, sheet_name="Tasa por mes", index=False)
    rates_poutcome.to_excel(writer, sheet_name="Tasa poutcome", index=False)
    rates_job.to_excel(writer, sheet_name="Tasa job", index=False)

print(f"Resultados guardados en: {OUTPUT_DIR.resolve()}")

## 11. Conclusion ejecutiva para direccion

1. El dataset contiene una clase positiva minoritaria: solo una parte reducida de clientes suscribe el deposito. Por ello, la compania no debe evaluar los modelos solo con accuracy; debe usar AUC, precision, recall, F1 y lift.

2. Para priorizar clientes antes de una campana, el modelo correcto es el mejor modelo **sin `duration`**, porque esa variable no existe antes de llamar. El escenario con `duration` sirve para diagnosticar la calidad o resultado de llamadas, no para decidir a quien contactar.

3. El modelo operativo permite ordenar clientes por probabilidad de conversion. La recomendacion es empezar la gestion por los deciles superiores de score, donde el lift es mayor que el promedio historico.

4. A nivel comercial, esto permite asignar mejores horarios, agentes mas experimentados o incentivos a los clientes con mayor propension, reduciendo contactos poco productivos.

5. Como siguiente paso, se recomienda validar el modelo con una campana piloto A/B: un grupo priorizado por score frente a un grupo gestionado con reglas tradicionales.